# Stage 4: Cleaning

Turn the raw transaction log into one clean, typed table by applying the drop list the
EDA produced. Every filter is shown with before and after counts plus the share of rows
and revenue it removes.

Prototyped here first, then refactored into `src/cleaning.py` and imported back to prove
the module reproduces the notebook exactly. Serving will call that same function.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_ROOT))

# Data directory
DATA_DIR = Path(PROJECT_ROOT, "data")
DATA_DIR.mkdir(exist_ok=True)

In [2]:
import polars as pl

from src.config import CLEAN_PARQUET, RAW_PARQUET
from src.logger import setup_logger

logger = setup_logger("02-cleaning")
pl.Config.set_tbl_rows(20)

logger.info("raw   : %s", RAW_PARQUET.name)
logger.info("output: %s", CLEAN_PARQUET.name)

18:17:26 | 02-cleaning | INFO | raw   : transactions_raw.parquet


18:17:26 | 02-cleaning | INFO | output: transactions_clean.parquet


## 1. Load raw and set up filter accounting

Loading the Stage 2 Parquet untouched. Also defining a small helper that reports every
filter the same way, because a drop with no number attached is not a documented drop.

Revenue is measured on gross absolute value (`|quantity * price|`) rather than net, so a
step that removes both a sale and its matching cancellation still shows its true size
instead of netting to zero.

In [3]:
raw = pl.read_parquet(RAW_PARQUET)
logger.info("raw shape: %s rows x %s columns", f"{raw.height:,}", raw.width)

GROSS = (pl.col("quantity") * pl.col("price")).abs()
start_rows = raw.height
start_gross = raw.select(GROSS.sum()).item()
audit = []


def step(name: str, before: pl.DataFrame, after: pl.DataFrame) -> pl.DataFrame:
    """Log one filter and record it for the summary table."""
    removed = before.height - after.height
    gross_removed = before.select(GROSS.sum()).item() - after.select(GROSS.sum()).item()
    audit.append({
        "step": name,
        "rows_before": before.height,
        "rows_removed": removed,
        "pct_rows": round(100 * removed / start_rows, 3),
        "pct_gross_revenue": round(100 * gross_removed / start_gross, 3),
    })
    logger.info("%-28s %9s -> %9s  (-%s rows, %.3f%% of gross revenue)",
                name, f"{before.height:,}", f"{after.height:,}",
                f"{removed:,}", 100 * gross_removed / start_gross)
    return after


logger.info("baseline gross revenue: %s", f"{start_gross:,.0f}")

18:17:26 | 02-cleaning | INFO | raw shape: 1,067,371 rows x 9 columns


18:17:26 | 02-cleaning | INFO | baseline gross revenue: 22,658,686


1,067,371 rows, gross revenue 22,658,686 as the denominator below.

## 2. Normalise before filtering

Trim whitespace (the EDA found `47503J ` with a trailing space) and derive
`is_cancellation` from the `C` prefix, not from `quantity < 0` (those 3,457 rows are
warehouse write-offs, not returns). Both must happen before any code-based filter.

In [4]:
STRING_COLS = ["invoice", "stock_code", "description", "country"]

for col in STRING_COLS:
    n = raw.filter(pl.col(col) != pl.col(col).str.strip_chars()).height
    if n:
        logger.info("%-12s %s rows change on trim", col, f"{n:,}")
        display(
            raw.filter(pl.col(col) != pl.col(col).str.strip_chars())
            .select(pl.col(col).alias("before"), pl.col(col).str.strip_chars().alias("after"))
            .unique().head(5)
        )

df = raw.with_columns(
    [pl.col(c).str.strip_chars().alias(c) for c in STRING_COLS]
).with_columns(
    pl.col("invoice").str.starts_with("C").alias("is_cancellation")
)

logger.info("distinct stock_code: %d -> %d", raw["stock_code"].n_unique(), df["stock_code"].n_unique())
logger.info("  '47503J' already existed untrimmed? %s",
            "47503J" in raw["stock_code"].implode().to_list()[0])
logger.info("is_cancellation true: %s (%.2f%%)",
            f"{df['is_cancellation'].sum():,}", 100 * df["is_cancellation"].mean())

18:17:26 | 02-cleaning | INFO | stock_code   1 rows change on trim


before,after
str,str
"""47503J ""","""47503J"""


18:17:26 | 02-cleaning | INFO | description  213,035 rows change on trim


before,after
str,str
"""GREEN CAT FLORAL CUSHION COVER…","""GREEN CAT FLORAL CUSHION COVER"""
"""MOODY GIRL DOOR HANGER ""","""MOODY GIRL DOOR HANGER"""
"""BLUE PAISLEY JOURNAL ""","""BLUE PAISLEY JOURNAL"""
"""SKULLS WRITING SET ""","""SKULLS WRITING SET"""
"""VINTAGE PINK TINSEL REEL ""","""VINTAGE PINK TINSEL REEL"""


18:17:26 | 02-cleaning | INFO | distinct stock_code: 5305 -> 5304


18:17:26 | 02-cleaning | INFO |   '47503J' already existed untrimmed? True


18:17:26 | 02-cleaning | INFO | is_cancellation true: 19,494 (1.83%)


`stock_code` changed on 1 row, as expected. Distinct codes went 5,305 to 5,304:
`47503J ` was a duplicate of an existing code, not a separate product. `description`
needed trimming on 213,035 rows, but it's dropped later so this costs nothing.
`is_cancellation` is 19,494 rows (1.83%), matching the EDA count.

## 3. The drop list

Four filters, each reported against the original 1,067,371 rows.

### 3a. `A`-prefixed invoices

Six rows of `Adjust bad debt` accounting, -147,614 net revenue. Not transactions.

In [5]:
df = step("drop A-prefix invoices", df,
          df.filter(~pl.col("invoice").str.starts_with("A")))

# Confirm the only invoice kinds left are numeric and C.
remaining = (
    df.select(
        pl.when(pl.col("invoice").str.contains(r"^\d+$")).then(pl.lit("numeric"))
        .when(pl.col("invoice").str.starts_with("C")).then(pl.lit("C prefix"))
        .otherwise(pl.lit("UNCLASSIFIED")).alias("kind")
    ).group_by("kind").len().sort("len", descending=True)
)
display(remaining)
assert "UNCLASSIFIED" not in remaining["kind"].to_list()

18:17:26 | 02-cleaning | INFO | drop A-prefix invoices       1,067,371 -> 1,067,365  (-6 rows, 0.749% of gross revenue)


kind,len
str,u32
"""numeric""",1047871
"""C prefix""",19494


### 3b. Non-product stock codes

Explicit list, not a regex: `DCGS*` and `SP1002` look like junk but are real products.
Checking what non-standard codes remain, as a check on the keep list.

In [6]:
NON_PRODUCT_CODES = frozenset({
    # shipping and carriage
    "POST", "DOT", "C2", "C3",
    # fees, adjustments, accounting
    "M", "m", "D", "S", "BANK CHARGES", "AMAZONFEE", "CRUK", "B", "ADJUST", "ADJUST2", "PADS",
    # internal test rows
    "TEST001", "TEST002",
    # gift vouchers: payment instruments, not products
    "GIFT", "gift_0001_10", "gift_0001_20", "gift_0001_30", "gift_0001_40", "gift_0001_50",
    "gift_0001_60", "gift_0001_70", "gift_0001_80", "gift_0001_90",
})

logger.info("codes on the drop list present in the data: %d of %d",
            df.filter(pl.col("stock_code").is_in(NON_PRODUCT_CODES))["stock_code"].n_unique(),
            len(NON_PRODUCT_CODES))

df = step("drop non-product codes", df,
          df.filter(~pl.col("stock_code").is_in(NON_PRODUCT_CODES)))

survivors = df.filter(~pl.col("stock_code").str.contains(r"^\d{5}\w*$"))
logger.info("non-standard codes kept: %d distinct, %s rows",
            survivors["stock_code"].n_unique(), f"{survivors.height:,}")
display(
    survivors.group_by("stock_code")
    .agg(pl.len().alias("rows"), pl.col("description").drop_nulls().first().alias("description"))
    .sort("rows", descending=True).head(10)
)

18:17:26 | 02-cleaning | INFO | codes on the drop list present in the data: 26 of 27


18:17:26 | 02-cleaning | INFO | drop non-product codes       1,067,365 -> 1,061,439  (-5,926 rows, 7.292% of gross revenue)


18:17:26 | 02-cleaning | INFO | non-standard codes kept: 35 distinct, 161 rows


stock_code,rows,description
str,u32,str
"""DCGS0058""",31,"""MISO PRETTY GUM"""
"""DCGSSGIRL""",25,"""update"""
"""DCGSSBOY""",23,"""update"""
"""DCGS0076""",15,"""SUNJAR LED NIGHT NIGHT LIGHT"""
"""DCGS0003""",14,"""BOXED GLASS ASHTRAY"""
"""DCGS0069""",6,"""OOH LA LA DOGS COLLAR"""
"""DCGS0004""",5,"""HAYNES CAMPER SHOULDER BAG"""
"""DCGS0066N""",4,"""NAVY CUDDLES DOG HOODIE"""
"""DCGS0072""",4,"""CAT CAMOUFLAGUE COLLAR"""


**`A` rows: 0.75% of gross revenue** from just 6 rows, large accounting entries.

**Non-product codes: 0.56% of rows but 7.29% of revenue.** Postage and fees carry real
money. Monetary features will measure product spend, not invoice value.

**26 of 27 codes matched**: `B` was already removed with the `A` invoices.

**35 non-standard codes survive** (161 rows), the `DCGS*`/`SP1002` products, as intended.

### 3c. Non-positive price

6,202 zero-price rows in the EDA, 98.9% unattributed. A genuine cancellation keeps a
positive price and flips quantity negative, so this cannot touch real returns. Asserting
that rather than trusting it.

In [7]:
doomed = df.filter(pl.col("price") <= 0)
logger.info("price <= 0: %s rows | %s are cancellations | %.1f%% unattributed",
            f"{doomed.height:,}", f"{doomed['is_cancellation'].sum():,}",
            100 * doomed["customer_id"].null_count() / doomed.height)

cancels_before = df["is_cancellation"].sum()
df = step("drop price <= 0", df, df.filter(pl.col("price") > 0))

lost = cancels_before - df["is_cancellation"].sum()
logger.info("cancellations lost to this filter: %d", lost)
assert lost == 0, "a price filter must never remove a real cancellation"

18:17:26 | 02-cleaning | INFO | price <= 0: 6,150 rows | 0 are cancellations | 99.0% unattributed


18:17:26 | 02-cleaning | INFO | drop price <= 0              1,061,439 -> 1,055,289  (-6,150 rows, 0.000% of gross revenue)


18:17:26 | 02-cleaning | INFO | cancellations lost to this filter: 0


6,150 rows removed, 0 cancellations, 99% unattributed, 0.000% of revenue: zero-priced
rows are worth zero by definition. Count differs from the EDA's 6,202 because 52 were
already removed as non-product codes.

### 3d. Leftover negative quantity

Backstop: after 3c, negative quantity should only mean cancellation. Expecting zero rows.

In [8]:
leftovers = df.filter((pl.col("quantity") < 0) & ~pl.col("is_cancellation"))
logger.info("negative quantity that is not a cancellation: %s rows", f"{leftovers.height:,}")
if leftovers.height:
    display(leftovers.select(
        "invoice", "stock_code", "description", "quantity", "price", "customer_id"
    ).head(10))

df = step("drop non-cancellation negatives", df,
          df.filter(pl.col("quantity").gt(0) | pl.col("is_cancellation")))

# After this point, negative quantity and is_cancellation must mean the same thing.
assert df.filter((pl.col("quantity") < 0) != pl.col("is_cancellation")).height == 0
logger.info("negative quantity and is_cancellation now agree on every row")

18:17:27 | 02-cleaning | INFO | negative quantity that is not a cancellation: 0 rows


18:17:27 | 02-cleaning | INFO | drop non-cancellation negatives 1,055,289 -> 1,055,289  (-0 rows, 0.000% of gross revenue)


18:17:27 | 02-cleaning | INFO | negative quantity and is_cancellation now agree on every row


Zero rows, as expected: all write-offs were zero-priced and already caught by 3c. The
filter stays anyway as an explicit guarantee. The assertion confirms `is_cancellation`
now agrees with quantity sign on every row, an invariant Stage 5 relies on.

## 4. Dtypes and derived columns

Cast `customer_id` to Int64, add `line_revenue = quantity * price` (negative on
cancellations by construction), drop `description` and `source_sheet` (duplicate notes
and a time-proxy leak risk). No duplicate-row filter: kept per the Stage 3 decision,
since duplicates don't affect invoice-level counts.

In [9]:
df = df.with_columns(
    pl.col("customer_id").cast(pl.Int64),
    (pl.col("quantity") * pl.col("price")).alias("line_revenue"),
).drop(["description", "source_sheet"], strict=False)

logger.info("columns: %s", df.columns)
logger.info("line_revenue: min %.2f | max %.2f | sum %.0f",
            df["line_revenue"].min(), df["line_revenue"].max(), df["line_revenue"].sum())

n_dupe = df.height - df.unique().height
logger.info("exact duplicate rows kept: %s (%.2f%%)", f"{n_dupe:,}", 100 * n_dupe / df.height)

18:17:27 | 02-cleaning | INFO | columns: ['invoice', 'stock_code', 'quantity', 'invoice_date', 'price', 'customer_id', 'country', 'is_cancellation', 'line_revenue']


18:17:27 | 02-cleaning | INFO | line_revenue: min -168469.60 | max 168469.60 | sum 19383662


18:17:27 | 02-cleaning | INFO | exact duplicate rows kept: 34,037 (3.23%)


`line_revenue` sums to 19,383,662, down from the raw gross of 22,658,686.

Duplicate count jumped from 12,133 to 34,037 after dropping `description`: rows that
only differed by a warehouse note are now identical. Doesn't change the keep decision,
just who counts as a duplicate.

## 5. Filter audit, end to end

In [10]:
audit_df = pl.DataFrame(audit)
display(audit_df)

total_removed = start_rows - df.height
logger.info("total: %s -> %s rows (-%s, -%.2f%%)",
            f"{start_rows:,}", f"{df.height:,}", f"{total_removed:,}",
            100 * total_removed / start_rows)
logger.info("gross revenue removed: %.2f%%", audit_df["pct_gross_revenue"].sum())

step,rows_before,rows_removed,pct_rows,pct_gross_revenue
str,i64,i64,f64,f64
"""drop A-prefix invoices""",1067371,6,0.001,0.749
"""drop non-product codes""",1067365,5926,0.555,7.292
"""drop price <= 0""",1061439,6150,0.576,0.0
"""drop non-cancellation negative…",1055289,0,0.0,0.0


18:17:27 | 02-cleaning | INFO | total: 1,067,371 -> 1,055,289 rows (-12,082, -1.13%)


18:17:27 | 02-cleaning | INFO | gross revenue removed: 8.04%


1.13% of rows dropped, 8.04% of gross revenue. Tracking revenue alongside rows is why
that gap is visible: non-product codes were 0.56% of rows but 7.29% of revenue.

## 6. Save, reload, verify

Not saving yet: the real save comes from the refactored function next, so the file on
disk is produced by the same code serving will use, not this ad hoc cell sequence.

In [11]:
logger.info("prototype clean shape: %s rows x %s columns", f"{df.height:,}", df.width)
logger.info("dtypes: %s", dict(df.schema))
logger.info("nulls remaining: %s", {c: n for c, n in zip(df.columns, df.null_count().row(0)) if n > 0})
logger.info("distinct customers: %s | distinct invoices: %s",
            f"{df['customer_id'].n_unique():,}", f"{df['invoice'].n_unique():,}")

18:17:27 | 02-cleaning | INFO | prototype clean shape: 1,055,289 rows x 9 columns


18:17:27 | 02-cleaning | INFO | dtypes: {'invoice': String, 'stock_code': String, 'quantity': Int64, 'invoice_date': Datetime(time_unit='us', time_zone=None), 'price': Float64, 'customer_id': Int64, 'country': String, 'is_cancellation': Boolean, 'line_revenue': Float64}


18:17:27 | 02-cleaning | INFO | nulls remaining: {'customer_id': 234722}


18:17:27 | 02-cleaning | INFO | distinct customers: 5,876 | distinct invoices: 46,921


1,055,289 rows, 9 columns, dtypes correct. Only null column is `customer_id` (234,722),
intentional: population filtering is Stage 5's job. Customers 5,942 to 5,876 and
invoices 53,628 to 46,921, both expected from the dropped rows.

## 7. Refactor into `src/cleaning.py`, prove equivalence

Same steps, now `clean_transactions()` plus `validate_clean()` for the invariants above.
Importing fresh and comparing against the prototype `df`: expecting an exact match.

In [12]:
from src.cleaning import clean_transactions

module_result = clean_transactions(pl.read_parquet(RAW_PARQUET))

same_shape = module_result.shape == df.shape
same_columns = module_result.columns == df.columns
prototype_sorted = df.sort(df.columns)
module_sorted = module_result.select(df.columns).sort(df.columns)
identical = prototype_sorted.equals(module_sorted)

logger.info("prototype shape %s vs module shape %s -> match: %s", df.shape, module_result.shape, same_shape)
logger.info("column order matches: %s", same_columns)
logger.info("row-for-row identical after sorting: %s", identical)

assert same_shape and identical, "module output diverged from the notebook prototype"

18:17:27 | cleaning | INFO | cleaned: 1,067,371 -> 1,055,289 rows (-1.13%)


18:17:27 | cleaning | INFO | validated: invariants hold on 1,055,289 rows


18:17:27 | 02-cleaning | INFO | prototype shape (1055289, 9) vs module shape (1055289, 9) -> match: True


18:17:27 | 02-cleaning | INFO | column order matches: True


18:17:27 | 02-cleaning | INFO | row-for-row identical after sorting: True


## 8. Save via the module, reload, verify

In [13]:
module_result.write_parquet(CLEAN_PARQUET)
logger.info("wrote %s (%.1f MB)", CLEAN_PARQUET.name, CLEAN_PARQUET.stat().st_size / 1024**2)

reloaded = pl.read_parquet(CLEAN_PARQUET)
assert reloaded.shape == module_result.shape
assert reloaded.schema == module_result.schema
assert reloaded.filter(pl.col("price") <= 0).height == 0
assert reloaded.filter(pl.col("invoice").str.starts_with("A")).height == 0
logger.info("reload verified: shape %s, dtypes match, invariants hold", reloaded.shape)
display(reloaded.head(3))

18:17:27 | 02-cleaning | INFO | wrote transactions_clean.parquet (6.2 MB)


18:17:27 | 02-cleaning | INFO | reload verified: shape (1055289, 9), dtypes match, invariants hold


invoice,stock_code,quantity,invoice_date,price,customer_id,country,is_cancellation,line_revenue
str,str,i64,datetime[μs],f64,i64,str,bool,f64
"""489434""","""85048""",12,2009-12-01 07:45:00,6.95,13085,"""United Kingdom""",false,83.4
"""489434""","""79323P""",12,2009-12-01 07:45:00,6.75,13085,"""United Kingdom""",false,81.0
"""489434""","""79323W""",12,2009-12-01 07:45:00,6.75,13085,"""United Kingdom""",false,81.0


## Summary

`data/processed/transactions_clean.parquet`: 1,055,289 rows, 9 columns (from 1,067,371
raw, -1.13% rows, -8.04% gross revenue). Logic lives in `src/cleaning.py`
(`clean_transactions`, `validate_clean`), proven identical to this notebook's prototype.

Dropped: `A`-prefixed invoices, non-product `stock_code` rows (explicit list), non-positive
price, description, source_sheet. Added: `is_cancellation`, `line_revenue`. Kept: null
`customer_id` (Stage 5's population filter), duplicate rows (don't affect invoice counts).

Open for Stage 5: pin `COUNTRY_VOCAB`, build the RFM and lifecycle features, apply the
population filter, construct the label.